# Wizualizacja obłożenia basenu MOSiR Łańcut

Ten notebook wczytuje dane zebrane przez GitHub Actions (plik `data/basen_dane.csv` w Twoim repozytorium) i pokazuje kilka wykresów:
1. liczba osób w czasie (cały zebrany okres),
2. heatmapa: dzień tygodnia x godzina — średnie obłożenie,
3. średnie obłożenie wg godziny,
4. konkretny dzień pod lupą.

**Zanim uruchomisz:** podmień `RAW_CSV_URL` poniżej na link do surowego pliku CSV w Twoim repo (przycisk „Raw” na GitHubie, np. `https://raw.githubusercontent.com/TWOJ_LOGIN/NAZWA_REPO/main/data/basen_dane.csv`).

In [ ]:
RAW_CSV_URL = "https://raw.githubusercontent.com/TWOJ_LOGIN/NAZWA_REPO/main/data/basen_dane.csv"

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df = pd.read_csv(RAW_CSV_URL)
df["timestamp_pl"] = pd.to_datetime(df["timestamp_pl"])
df["data"] = pd.to_datetime(df["data"]).dt.date
df = df.sort_values("timestamp_pl")
df["dzien_tygodnia"] = pd.to_datetime(df["data"]).dt.day_name()
print(f"Wczytano {len(df)} odczytów, od {df['timestamp_pl'].min()} do {df['timestamp_pl'].max()}")
df.tail()

## 1. Liczba osób w czasie (cały okres)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df["timestamp_pl"], df["osoby"], marker=".", markersize=3, linewidth=0.8)
ax.set_title("Liczba osób na basenie w czasie")
ax.set_xlabel("Data i godzina")
ax.set_ylabel("Liczba osób")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m %H:%M"))
fig.autofmt_xdate()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Heatmapa: dzień x godzina (średnie obłożenie)

In [ ]:
pivot = df.pivot_table(index="data", columns="godzina", values="osoby", aggfunc="mean")

fig, ax = plt.subplots(figsize=(14, max(4, 0.35 * len(pivot))))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([str(d) for d in pivot.index])
ax.set_xlabel("Godzina")
ax.set_ylabel("Dzień")
ax.set_title("Średnia liczba osób wg dnia i godziny")
fig.colorbar(im, ax=ax, label="Śr. liczba osób")
plt.tight_layout()
plt.show()

## 3. Średnie obłożenie wg godziny (wszystkie dni razem)

In [ ]:
srednia_godzinowa = df.groupby("godzina")["osoby"].mean().reindex(range(6, 22))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(srednia_godzinowa.index, srednia_godzinowa.values, color="#2E86AB")
ax.set_title("Średnia liczba osób na basenie wg godziny")
ax.set_xlabel("Godzina")
ax.set_ylabel("Średnia liczba osób")
ax.set_xticks(range(6, 22))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Konkretny dzień pod lupą
Zmień `WYBRANY_DZIEN` na interesującą Cię datę (format `YYYY-MM-DD`).

In [ ]:
import datetime as dt

WYBRANY_DZIEN = str(df["data"].max())  # domyślnie: ostatni zebrany dzień

dzien_df = df[df["data"] == dt.datetime.strptime(WYBRANY_DZIEN, "%Y-%m-%d").date()]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dzien_df["timestamp_pl"], dzien_df["osoby"], marker="o")
ax.set_title(f"Liczba osób na basenie — {WYBRANY_DZIEN}")
ax.set_xlabel("Godzina")
ax.set_ylabel("Liczba osób")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Średnie obłożenie dla wybranego dnia tygodnia
Ustaw `WYBRANY_DZIEN_TYGODNIA` na jeden z: `Monday`, `Tuesday`, `Wednesday`, `Thursday`, `Friday`, `Saturday`, `Sunday`.
Wykres pokazuje średnią liczbę osób w każdej godzinie (np. same niedziele), a zacieniowany pas — zakres min–max między poszczególnymi tygodniami, żeby było widać, jak bardzo obłożenie się waha.

In [ ]:
WYBRANY_DZIEN_TYGODNIA = "Sunday"  # np. Sunday, Monday, Saturday...

df_dzien = df[df["dzien_tygodnia"] == WYBRANY_DZIEN_TYGODNIA]

if df_dzien.empty:
    print(f"Brak jeszcze danych dla dnia: {WYBRANY_DZIEN_TYGODNIA}")
else:
    stat_godzinowa = df_dzien.groupby("godzina")["osoby"].agg(["mean", "min", "max", "count"]).reindex(range(6, 22))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(stat_godzinowa.index, stat_godzinowa["mean"], marker="o", color="#2E86AB", label="średnia")
    ax.fill_between(
        stat_godzinowa.index,
        stat_godzinowa["min"],
        stat_godzinowa["max"],
        alpha=0.2,
        color="#2E86AB",
        label="zakres min–max",
    )
    ax.set_title(f"Średnie obłożenie basenu — {WYBRANY_DZIEN_TYGODNIA} (liczba tygodni: {int(df_dzien['data'].nunique())})")
    ax.set_xlabel("Godzina")
    ax.set_ylabel("Liczba osób")
    ax.set_xticks(range(6, 22))
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Porownanie kilku dni tygodnia na jednym wykresie
Np. porównaj niedzielę z poniedziałkiem, albo cały tydzień naraz — wpisz interesujące Cię dni do listy `DNI_DO_POROWNANIA`.

In [ ]:
DNI_DO_POROWNANIA = ["Monday", "Sunday"]  # dowolna lista dni po angielsku

fig, ax = plt.subplots(figsize=(10, 5))
for dzien in DNI_DO_POROWNANIA:
    df_d = df[df["dzien_tygodnia"] == dzien]
    if df_d.empty:
        continue
    srednia = df_d.groupby("godzina")["osoby"].mean().reindex(range(6, 22))
    ax.plot(srednia.index, srednia.values, marker="o", label=dzien)

ax.set_title("Porównanie średniego obłożenia wg dnia tygodnia")
ax.set_xlabel("Godzina")
ax.set_ylabel("Średnia liczba osób")
ax.set_xticks(range(6, 22))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()